# TCC — Aplicação de modelos de machine learning na previsão do CDI
## Seção 4.4 — Interpretabilidade: análise SHAP

Notebook auxiliar à seção 4.4 do TCC. Reproduz o pipeline da 4.2/4.3, computa valores SHAP no treino e no teste, e gera as figuras 11, 12 e 13.

**Saídas:**
- Tabela 11 — top features por importância média absoluta de SHAP
- Figura 11 — beeswarm SHAP (top 20 features) + barra agrupada por categoria
- Figura 12 — waterfall SHAP em três datas representativas (02/01, 18/03 e 27/03/2026)
- Figura 13 — contribuição SHAP por grupo de variáveis ao longo do teste

**Pré-requisitos** (arquivos no ambiente):
- `modelo_xgb_final.joblib` — gerado pelo notebook de treino
- `dados_modelo_tcc_cdi.xlsx`
- `resultados_43_consolidado.xlsx` — usado para auditar que `model.predict(X_teste)` reproduz `Previsao_ML`


## 0. Setup do ambiente

In [ ]:
# Instala SHAP se ainda não estiver disponível (Colab já vem com xgboost, pandas, sklearn, matplotlib).
import importlib
if importlib.util.find_spec("shap") is None:


In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import xgboost as xgb
import shap

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
sns.set_style("whitegrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

print(f"xgboost: {xgb.__version__}")
print(f"shap:    {shap.__version__}")
print(f"numpy:   {np.__version__}")
print(f"pandas:  {pd.__version__}")


In [ ]:
# Upload dos arquivos no Colab.
# Se estiver rodando localmente, comente este bloco e coloque os arquivos no mesmo diretório do notebook.
try:
    print("Faça upload de: modelo_xgb_final.joblib, "
          "dados_modelo_tcc_cdi.xlsx, "
          "resultados_43_consolidado.xlsx.")
    print("Arquivos disponíveis:", list(uploaded.keys()))
except ImportError:
    print("Ambiente local — coloque os arquivos no diretório de trabalho.")


In [ ]:
# Parâmetros do experimento — espelham a 4.2 e a 4.3
PATH_MODEL    = "modelo_xgb_final.joblib"
PATH_CDI      = "dados/dados_modelo_tcc_cdi.xlsx"
PATH_RES_43   = "dados/resultados_43_consolidado.xlsx"

DATA_TREINO_INI = pd.Timestamp("2023-01-02")
DATA_TREINO_FIM = pd.Timestamp("2025-11-28")
DATA_TESTE_INI  = pd.Timestamp("2026-01-02")
DATA_TESTE_FIM  = pd.Timestamp("2026-03-31")

H = 22  # horizonte de previsão (dias úteis)

# Datas-alvo da análise local (4.4.2) — escolhidas para iluminar o viés direcional de 4.3.3
DATAS_LOCAIS = [pd.Timestamp("2026-01-02"),
                pd.Timestamp("2026-03-18"),
                pd.Timestamp("2026-03-27")]


## 4.4.0 — Carregamento e validação do pipeline

Esta seção carrega o modelo final salvo pela 4.2, reconstrói as bases de treino e teste com a mesma lógica usada na 4.3, e audita a consistência das previsões antes de prosseguir.

In [ ]:
# Carrega o modelo final e as colunas que ele espera
artefatos = joblib.load(PATH_MODEL)
model         = artefatos["model"]
feature_cols  = artefatos["feature_cols"]
best_params   = artefatos["best_params"]

print(f"Modelo:           {type(model).__name__}")
print(f"# features:       {len(feature_cols)}")
print(f"Hiperparâmetros:  {best_params}")

# Carrega a base consolidada e reconstrói o alvo (idêntico à 4.2)
df = pd.read_excel(PATH_CDI, sheet_name="Diario_Consolidado")
df["Data"] = pd.to_datetime(df["Data"])
df = df.sort_values("Data").reset_index(drop=True)

df["y_22d"] = (
    df["CDI"].shift(-1).rolling(window=H, min_periods=H).mean().shift(-(H - 1))
)

# Treino (mesma máscara da 4.2)
mask_treino = (df["Data"] >= DATA_TREINO_INI) & (df["Data"] <= DATA_TREINO_FIM)
df_treino = df.loc[mask_treino].dropna(subset=["y_22d"]).reset_index(drop=True)
X_treino     = df_treino[feature_cols]
y_treino     = df_treino["y_22d"]
dates_treino = df_treino["Data"]

# Teste (mesma máscara da 4.3)
mask_teste = (df["Data"] >= DATA_TESTE_INI) & (df["Data"] <= DATA_TESTE_FIM)
df_teste = df.loc[mask_teste].reset_index(drop=True)
X_teste      = df_teste[feature_cols].copy()
y_real_teste = df_teste["y_22d"].copy()
dates_teste  = df_teste["Data"].copy()

print(f"\nTreino: {len(df_treino)} obs ({dates_treino.min().date()} a {dates_treino.max().date()})")
print(f"Teste:  {len(df_teste)} obs ({dates_teste.min().date()} a {dates_teste.max().date()})")


In [ ]:
# Audit: model.predict(X_teste) deve reproduzir exatamente a coluna Previsao_ML salva pela 4.3
res43 = pd.read_excel(PATH_RES_43, sheet_name="Sheet1")
res43["Data"] = pd.to_datetime(res43["Data"])

y_pred_teste = model.predict(X_teste)
audit = pd.DataFrame({
    "Data":           dates_teste.values,
    "Previsao_atual": y_pred_teste,
})
audit = audit.merge(res43[["Data", "Previsao_ML"]], on="Data", how="inner")
audit["dif"] = (audit["Previsao_atual"] - audit["Previsao_ML"]).abs()

dif_max = audit["dif"].max()
print("Audit predict(X_teste) vs Previsao_ML da 4.3:")
print(f"  Datas comparadas: {len(audit)}")
print(f"  Diferença máxima: {dif_max:.2e}")
assert dif_max < 1e-5, "Reconstrução do pipeline divergiu da 4.3 — verificar antes de prosseguir."
print("  OK Pipeline reconstruído corretamente. Pode prosseguir.")


## 4.4.0.a — Cálculo dos valores SHAP

Aplica `shap.TreeExplainer` ao XGBoost treinado e produz a decomposição aditiva (`base_value + Σ SHAP_i = previsão`) tanto sobre o conjunto de treino quanto sobre o de teste. A propriedade aditiva é validada por uma asserção numérica antes de prosseguirmos para as figuras.

In [ ]:
# TreeExplainer — eficiente para XGBoost e exato para previsões de árvore (Lundberg et al., 2020)
explainer = shap.TreeExplainer(model)

# Explanation objects (carregam values, base_values e data)
expl_treino = explainer(X_treino)
expl_teste  = explainer(X_teste)

shap_values_treino = expl_treino.values        # shape (n_train, n_features)
shap_values_teste  = expl_teste.values         # shape (n_test, n_features)
base_value         = float(expl_treino.base_values[0])

print(f"shap_values_treino: shape {shap_values_treino.shape}")
print(f"shap_values_teste:  shape {shap_values_teste.shape}")
print(f"base_value (E[f(X)] no treino): {base_value:.4f} p.p.")

# Validação da propriedade aditiva: base + soma de SHAP por feature == previsão do modelo
recon_treino = base_value + shap_values_treino.sum(axis=1)
pred_treino  = model.predict(X_treino)
dif_treino_max = float(np.abs(recon_treino - pred_treino).max())

recon_teste = base_value + shap_values_teste.sum(axis=1)
pred_teste  = model.predict(X_teste)
dif_teste_max = float(np.abs(recon_teste - pred_teste).max())

print("\nValidação aditiva (base + Σ SHAP == predict):")
print(f"  Treino — diferença máx: {dif_treino_max:.2e}")
print(f"  Teste  — diferença máx: {dif_teste_max:.2e}")
assert dif_treino_max < 1e-4 and dif_teste_max < 1e-4
print("  OK Decomposição aditiva válida.")


In [ ]:
# Definição dos grupos de variáveis — espelha a taxonomia de 3.4 (cinco grupos) +
# separação explícita entre "calendário de eventos" e "comportamentais",
# conforme discutido para a 4.4.

GRUPOS = {
    "Autorregressivo": [
        "CDI", "CDI_lag1", "CDI_lag5", "CDI_lag22",
        "CDI_media22d", "CDI_volat22d",
    ],
    "Focus": [c for c in feature_cols if c.startswith("Focus_")],
    "Macro": [
        "Selic_Over", "Selic_Meta",
        "IPCA", "IPCA_15", "IPCA_Nucleo_MS", "IPCA_Nucleo_EX0",
        "IPCA_Nucleo_EX3", "IPCA_Servicos",
        "IBC_Br", "Producao_Industrial", "Vendas_Varejo",
    ],
    "Mercado": [
        "USD_BRL_Compra", "USD_BRL_Venda", "Ibovespa", "Brent_Oil", "WTI_Oil",
        "Soja_CBOT", "SP500", "DXY_Dolar", "CDS_Brasil", "Ibov_Retorno",
        "USD_BRL_Compra_lag1", "USD_BRL_Compra_lag5", "USD_BRL_Compra_lag22",
        "USD_BRL_Compra_media22d", "USD_BRL_Compra_volat22d",
        "Ibovespa_lag1", "Ibovespa_lag5", "Ibovespa_lag22",
        "Ibovespa_media22d", "Ibovespa_volat22d",
    ],
    "Comportamental": [
        "Surpresa_IPCA",
        "Copom_Surpresa_Direcao_5du", "Copom_Surpresa_Magnitude_5du",
        "Copom_Surpresa_Direcao_22du", "Copom_Surpresa_Magnitude_22du",
        "Revisao_Focus_Selic", "Revisao_Focus_IPCA",
        "Ibov_Cauda_Inferior", "Ibov_Cauda_Superior",
    ],
    "Calendário": ["Copom_Dia", "Copom_Semana", "Copom_PreReuniao"],
}

# Mapeamento inverso: feature → grupo
feat2grupo = {f: g for g, fs in GRUPOS.items() for f in fs}
nao_classificadas = [f for f in feature_cols if f not in feat2grupo]
print(f"# features classificadas: {sum(len(v) for v in GRUPOS.values())}")
print(f"# features no modelo:     {len(feature_cols)}")
print(f"# não-classificadas:      {len(nao_classificadas)}  {nao_classificadas}")

for g, fs in GRUPOS.items():
    print(f"  {g:18s} | {len(fs):3d} features")


## 4.4.1 — Importância global das features

A importância global é calculada como `mean(|SHAP|)` por feature sobre o conjunto de **treino**, padrão na literatura de interpretabilidade (Lundberg & Lee, 2017). Diferente da correlação de Pearson, essa medida captura contribuições mediadas por interações entre variáveis — exatamente o que árvores capturam e o que correlações lineares não capturam.

A análise é apresentada em três cortes:

1. **Tabela 11** — top 20 features por mean(|SHAP|), com a correlação linear de cada uma com o CDI no treino (mesma definição da Figura 6) ao lado. A coluna `Delta_Rank` mostra quanto a feature subiu (positivo) ou caiu (negativo) no ranking quando trocamos correlação por SHAP.
2. **Figura 11** — beeswarm SHAP das 20 mais importantes (magnitude e direção da contribuição).
3. **Figura 12** — agregação por grupo de variáveis (espelha a taxonomia de 3.4).

In [ ]:
# mean(|SHAP|) sobre o treino — métrica padrão de importância global
imp_abs = np.abs(shap_values_treino).mean(axis=0)
imp_df = pd.DataFrame({
    "Feature":     feature_cols,
    "Grupo":       [feat2grupo.get(f, "—") for f in feature_cols],
    "MeanAbsSHAP": imp_abs,
})

# Correlação com o CDI no período de treinamento (mesma definição da Figura 6)
cdi_treino = df_treino["CDI"]
correl = X_treino.apply(lambda s: s.corr(cdi_treino))
imp_df["Correl_CDI"] = imp_df["Feature"].map(correl)

imp_df = imp_df.sort_values("MeanAbsSHAP", ascending=False).reset_index(drop=True)
imp_df["Rank_SHAP"] = imp_df.index + 1

# Rank pela correlação em módulo, para identificar features que "sobem" com SHAP
rank_corr = imp_df.assign(absC=imp_df["Correl_CDI"].abs()).sort_values("absC", ascending=False)
rank_corr["Rank_Corr"] = range(1, len(rank_corr) + 1)
imp_df = imp_df.merge(rank_corr[["Feature", "Rank_Corr"]], on="Feature")
imp_df["Delta_Rank"] = imp_df["Rank_Corr"] - imp_df["Rank_SHAP"]
imp_df = imp_df.sort_values("Rank_SHAP").reset_index(drop=True)

print("Tabela 11 — top 20 por mean(|SHAP|):")
print(imp_df.head(20).to_string(index=False, float_format=lambda x: f"{x:.4f}"))

imp_df.to_excel("4_4_tabela11_importancia_global.xlsx", index=False)


In [ ]:
# Figura 11 — beeswarm SHAP do treino (top 20)
plt.figure()
shap.summary_plot(
    shap_values_treino,
    X_treino,
    feature_names=feature_cols,
    max_display=20,
    show=False,
    plot_size=(9, 8),
)
plt.title("Figura 11 — Importância SHAP (beeswarm, top 20) — período de treino",
          fontsize=11, loc="left")
plt.tight_layout()
plt.savefig("4_4_figura11_beeswarm.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# Figura 12 — importância agregada por grupo (mean(|SHAP|) somado por grupo)
imp_grupo = (
    imp_df.groupby("Grupo")["MeanAbsSHAP"]
          .sum()
          .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(8, 4))
imp_grupo.plot(kind="barh", color="#3a6ea5", ax=ax)
ax.set_xlabel("Soma de mean(|SHAP|) sobre o grupo (p.p.)")
ax.set_ylabel("")
ax.set_title("Figura 12 — Importância SHAP agregada por grupo de variáveis",
             fontsize=11, loc="left")
for i, v in enumerate(imp_grupo.values):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig("4_4_figura12_grupos.png", dpi=200, bbox_inches="tight")
plt.show()

print("\nContribuição por grupo (mean(|SHAP|) somado):")
print(imp_grupo.sort_values(ascending=False).to_string())


## 4.4.2 — Análise local: como o modelo decide em datas específicas

Três datas representativas, escolhidas para iluminar o **viés direcional** diagnosticado em 4.3.3. Cada uma recebe sua própria figura (Figuras 13, 14 e 15):

| Data | Contexto | Erro do XGBoost esperado | Figura |
|------|----------|--------------------------|--------|
| 02/01/2026 | Primeiro dia útil do teste; CDI realizado em 14,90% | Subestimação (≈ +0,14 p.p.) | Figura 13 |
| 18/03/2026 | Dia do corte de 25 pb (Selic 15,00% → 14,75%) | Transição | Figura 14 |
| 27/03/2026 | Última observação avaliável; CDI realizado em 14,65% | Superestimação (≈ −0,10 p.p.) | Figura 15 |

Para cada data: (i) waterfall SHAP completo, (ii) contexto com o realizado, a previsão do modelo e a taxa implícita da DI, (iii) tabela com as 10 maiores contribuições em valor absoluto.

In [ ]:
def contexto_data(t):
    """Retorna previsão, realizado e DI implícita para a data t do teste."""
    linha = res43[res43["Data"] == pd.Timestamp(t)]
    if len(linha) == 0:
        return None
    r = linha.iloc[0]
    return {
        "Data":          pd.Timestamp(r["Data"]).date(),
        "CDI_realizado": r.get("CDI_realizado", np.nan),
        "Previsao_ML":   r.get("Previsao_ML", np.nan),
        "DI_Implicita":  r.get("DI_Implicita", np.nan),
        "Erro_ML":       r.get("Erro_ML", np.nan),
        "Erro_DI":       r.get("Erro_DI", np.nan),
    }

def waterfall_data(t, numero_figura, max_display=15):
    """Plota waterfall SHAP para a data t do teste e retorna a tabela de contribuições.

    Parâmetros
    ----------
    t : str ou Timestamp
        Data do teste a explicar.
    numero_figura : int
        Número da figura (13, 14 ou 15), usado no título e no nome do arquivo.
    max_display : int
        Quantas features exibir no waterfall.
    """
    idx = np.where(dates_teste.values == np.datetime64(pd.Timestamp(t)))[0]
    if len(idx) == 0:
        print(f"Data {t} não está no teste.")
        return None
    i = int(idx[0])
    expl_i = expl_teste[i]

    ctx = contexto_data(t)
    print(f"=== {ctx['Data']} ===")
    print(f"  CDI realizado:    {ctx['CDI_realizado']:.4f}% a.a.")
    print(f"  Previsão XGBoost: {ctx['Previsao_ML']:.4f}% a.a.  (erro = {ctx['Erro_ML']:+.4f} p.p.)")
    print(f"  DI implícita:     {ctx['DI_Implicita']:.4f}% a.a.  (erro = {ctx['Erro_DI']:+.4f} p.p.)")
    print(f"  Base value SHAP:  {float(expl_i.base_values):.4f}% a.a.")
    print(f"  Soma de SHAP:     {float(expl_i.values.sum()):+.4f} p.p.")

    plt.figure(figsize=(8, 6))
    shap.plots.waterfall(expl_i, max_display=max_display, show=False)
    plt.title(f"Figura {numero_figura} — Waterfall SHAP em {ctx['Data']}",
              fontsize=11, loc="left")
    plt.tight_layout()
    safe_d = str(ctx['Data']).replace("-", "")
    plt.savefig(f"4_4_figura{numero_figura}_waterfall_{safe_d}.png",
                dpi=200, bbox_inches="tight")
    plt.show()

    contrib = pd.DataFrame({
        "Feature": feature_cols,
        "Valor":   X_teste.iloc[i].values,
        "SHAP":    expl_i.values,
        "Grupo":   [feat2grupo.get(f, "—") for f in feature_cols],
    })
    contrib["AbsSHAP"] = contrib["SHAP"].abs()
    top = contrib.sort_values("AbsSHAP", ascending=False).head(10)
    print("\n  Top 10 contribuições (em valor absoluto):")
    print(top[["Feature", "Grupo", "Valor", "SHAP"]].to_string(index=False,
          float_format=lambda x: f"{x:.4f}"))
    return contrib


In [ ]:
contrib_0102 = waterfall_data("2026-01-02", numero_figura=13)


In [ ]:
contrib_1803 = waterfall_data("2026-03-18", numero_figura=14)


In [ ]:
contrib_2703 = waterfall_data("2026-03-27", numero_figura=15)


## 4.4.3 — Estabilidade temporal das contribuições

A contribuição SHAP de cada grupo é somada (com sinal preservado) para cada data do teste. Isso responde a duas perguntas:

1. **Composição.** O grupo que mais contribui para a previsão muda ao longo do trimestre? Em particular, as variáveis comportamentais e de calendário ganham peso na vizinhança das reuniões do Copom (28/01 e 18/03)?
2. **Direção.** Como as contribuições se reorganizam em torno do corte de 25 pb? A queda na previsão do modelo após 18/03 é dominada por qual grupo?

A **Figura 16** mostra a soma assinada por grupo ao longo do tempo, com linhas verticais marcando as reuniões do Copom (28/01 e 18/03). Para contexto sobre o comportamento das previsões nesse mesmo período, basta cruzar com a Figura 9 já apresentada em 4.3.3.

In [ ]:
# Soma assinada de SHAP por grupo em cada data do teste
shap_test_df = pd.DataFrame(shap_values_teste, columns=feature_cols, index=dates_teste.values)

contrib_grupo_teste = pd.DataFrame(index=dates_teste.values)
for grupo, fs in GRUPOS.items():
    fs_validas = [f for f in fs if f in shap_test_df.columns]
    contrib_grupo_teste[grupo] = shap_test_df[fs_validas].sum(axis=1).values

# Confere que soma por grupo + base == previsão
recon = contrib_grupo_teste.sum(axis=1).values + base_value
dif_max = float(np.abs(recon - pred_teste).max())
print(f"Validação (soma por grupo + base == predict): dif máx {dif_max:.2e}")
assert dif_max < 1e-4

print("\nContribuição média por grupo no teste (p.p., sinal preservado):")
print(contrib_grupo_teste.mean().sort_values(ascending=False).to_string())
print("\nContribuição em |SHAP| média por grupo no teste (p.p.):")
print(contrib_grupo_teste.abs().mean().sort_values(ascending=False).to_string())

contrib_grupo_teste.to_excel("4_4_contribuicao_grupo_teste.xlsx", index_label="Data")


In [ ]:
# Figura 16 — contribuição SHAP por grupo ao longo do teste (assinada)
copom_no_teste = [pd.Timestamp("2026-01-28"), pd.Timestamp("2026-03-18")]
cores = {
    "Autorregressivo": "#1f4e79",
    "Focus":           "#c0504d",
    "Macro":           "#9bbb59",
    "Mercado":         "#8064a2",
    "Comportamental":  "#f79646",
    "Calendário":      "#7f7f7f",
}

fig, ax = plt.subplots(figsize=(10, 5.5))
for g in contrib_grupo_teste.columns:
    ax.plot(contrib_grupo_teste.index, contrib_grupo_teste[g],
            label=g, color=cores.get(g, None), lw=1.4)
for c in copom_no_teste:
    ax.axvline(c, color="black", lw=0.8, ls="--", alpha=0.6)
ax.axhline(0, color="black", lw=0.6)
ax.set_ylabel("Contribuição SHAP (p.p.)")
ax.set_xlabel("Data")
ax.set_title("Figura 16 — Contribuição SHAP por grupo ao longo do teste (assinada)",
             fontsize=11, loc="left")
ax.legend(loc="best", ncol=3, fontsize=9, frameon=True)

plt.tight_layout()
plt.savefig("4_4_figura16_grupos_no_tempo.png", dpi=200, bbox_inches="tight")
plt.show()


## Encerramento — persistência de artefatos

In [ ]:
# Persiste resumos para uso na redação
joblib.dump({
    "shap_values_treino": shap_values_treino,
    "shap_values_teste":  shap_values_teste,
    "base_value":         base_value,
    "feature_cols":       feature_cols,
    "GRUPOS":             GRUPOS,
    "dates_treino":       dates_treino.values,
    "dates_teste":        dates_teste.values,
}, "shap_artefatos.joblib")

artefatos_44 = [
    "4_4_tabela11_importancia_global.xlsx",
    "4_4_contribuicao_grupo_teste.xlsx",
    "shap_artefatos.joblib",
    "4_4_figura11_beeswarm.png",
    "4_4_figura12_grupos.png",
] + [f"4_4_figura{n}_waterfall_{str(pd.Timestamp(d).date()).replace('-','')}.png"
     for n, d in zip([13, 14, 15], DATAS_LOCAIS)] + [
    "4_4_figura16_grupos_no_tempo.png",
]

print("Artefatos gerados:")
for a in artefatos_44:
    existe = os.path.exists(a)
    print(f"  [{'OK' if existe else '--'}] {a}")

# Download em ambiente Colab
try:
    for a in artefatos_44:
        if os.path.exists(a):
            files.download(a)
except ImportError:
    pass


---

**Próximos passos (redação da 4.4):**

- **4.4.1** — usar `4_4_tabela11_importancia_global.xlsx` para descrever o top 20, `4_4_figura11_beeswarm.png` como Figura 11 e `4_4_figura12_grupos.png` como Figura 12. Comparar `Rank_SHAP` com `Rank_Corr` na coluna `Delta_Rank`: features com Delta positivo grande são as que sobem no ranking SHAP (sinal de uso via interações).
- **4.4.2** — três waterfalls como Figuras 13, 14 e 15 (`4_4_figura13_waterfall_20260102.png`, `4_4_figura14_waterfall_20260318.png`, `4_4_figura15_waterfall_20260327.png`), acompanhadas das três tabelas de top contribuições impressas no notebook. Iluminar o viés direcional discutido em 4.3.3.
- **4.4.3** — `4_4_figura16_grupos_no_tempo.png` como Figura 16, e o `4_4_contribuicao_grupo_teste.xlsx` para extrair médias por sub-período (pré-Copom de janeiro, entre Copom, pós-Copom de março).
